# Lab: Einstein Summation in PyTorch

`torch.einsum` is a single function that expresses any multilinear operation on tensors — dot products, matrix products, traces, outer products, and batched variants — using a compact index-contraction notation. Every formula you studied in the Matrix Algebra readings can be written as a single einsum call.

This lab works through those formulas in order, ending with three highlights from Reading 5: the spectral decomposition $A = Q\Lambda Q'$ written as a sum of rank-1 outer products, the Eckart–Young truncated SVD $A_r = \sum_{i=1}^r \sigma_i \mathbf{u}_i\mathbf{v}_i'$, and a full PCA pipeline.

| Section | Reading |
|---------|--------|
| Unary operations (transpose, trace, diagonal, sums) | r2 Matrix Operations |
| Binary products (dot, outer, matvec, matmul, $X'X$) | r2 Matrix Operations |
| Batch operations | r1 / r2 |
| Quadratic forms and positive definiteness | r5 |
| **Spectral decomposition** $A = \sum_k \lambda_k \mathbf{q}_k\mathbf{q}_k'$ | **r5** |
| **SVD and truncated low-rank approximation** | **r5** |
| **PCA: covariance → eigendecomposition → projection** | **r5** |
| 3DGS batch covariance $\Sigma = R\,\text{diag}(s^2)\,R'$ | r5 + 3DGS course |
| Performance benchmark | — |

In [ ]:
import torch
import time
import numpy as np

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__}')
print(f'Device : {device}')

## Notation Primer

An einsum string has the form `"inputs -> output"`. Each letter names an axis. A letter that appears in inputs but **not** the output is **contracted** (summed over). A letter that appears in multiple inputs must match in size — it is the dimension being multiplied across.

| Math | Einsum string | Contracted index |
|------|--------------|------------------|
| $A^\top$ | `'ij->ji'` | none |
| $\text{tr}(A) = \sum_i a_{ii}$ | `'ii->'` | $i$ (diagonal) |
| $\mathbf{a}'\mathbf{b} = \sum_i a_i b_i$ | `'i,i->'` | $i$ |
| $\mathbf{a}\mathbf{b}' = a_i b_j$ | `'i,j->ij'` | none |
| $c_{ik} = \sum_j a_{ij} b_{jk}$ | `'ij,jk->ik'` | $j$ |
| $\sum_{i=1}^n \lambda_i \mathbf{q}_i\mathbf{q}_i'$ | `'k,ik,jk->ij'` | $k$ |

## Section 1 — Unary Operations &nbsp; `📖 r2`

From **Reading r2**: transposition swaps row and column indices ($b_{ij} = a_{ji}$); the trace sums the diagonal; row/column sums collapse one index. All of these are unary einsum calls — one input, one output, no new computation besides index permutation and optional contraction.

In [ ]:
A = torch.arange(1, 17, dtype=torch.float32).reshape(4, 4)
print('A =\n', A)

# r2: B = A'  ↔  b_{ij} = a_{ji}
AT  = torch.einsum('ij->ji', A)
print('\nTranspose  match A.T:', torch.allclose(AT, A.T))

# r2: sum all elements
s   = torch.einsum('ij->', A)
print(f'Sum all    {s.item():.0f}  match: {torch.allclose(s, A.sum())}')

# r2: row sums  (sum over column index j)
rs  = torch.einsum('ij->i', A)
print(f'Row sums   {rs.tolist()}  match: {torch.allclose(rs, A.sum(dim=1))}')

# r2: column sums  (sum over row index i)
cs  = torch.einsum('ij->j', A)
print(f'Col sums   {cs.tolist()}  match: {torch.allclose(cs, A.sum(dim=0))}')

# r5: tr(A) = Σ λ_i  — trace = sum of eigenvalues
tr  = torch.einsum('ii->', A)
print(f'Trace      {tr.item():.0f}  match: {torch.allclose(tr, torch.trace(A))}')

# diagonal extraction (needed for spectral section later)
d   = torch.einsum('ii->i', A)
print(f'Diagonal   {d.tolist()}  match: {torch.allclose(d, torch.diag(A))}')

## Section 2 — Binary Products &nbsp; `📖 r2`

From **Reading r2**:
- **Inner product**: $\mathbf{a}'\mathbf{b} = \sum_i a_i b_i$ — orthogonality when zero
- **Outer product**: $\mathbf{a}\mathbf{b}'$ — rank-1 matrix; the building block of the spectral decomposition
- **Matrix product**: $c_{ik} = \sum_j a_{ij} b_{jk}$ — index $j$ is contracted
- **Gram matrix** $X'X$: element $[X'X]_{jk}$ is the inner product of columns $j$ and $k$ of $X$ — the foundation of the sample covariance matrix

In [ ]:
a = torch.tensor([1., 2., 3.])
b = torch.tensor([4., 5., 6.])
M = torch.randn(3, 5)
N = torch.randn(5, 4)
P = torch.randn(3, 5)   # same shape as M, for Frobenius product

# r2: inner product  a'b = Σ_i a_i b_i
dot = torch.einsum('i,i->', a, b)
print(f"Inner product  a'b = {dot.item():.1f}  match: {torch.allclose(dot, a @ b)}")

# r2: outer product  ab' — rank-1 matrix (each column of b scales a)
outer = torch.einsum('i,j->ij', a, b)
print(f'Outer product  shape {outer.shape}  match: {torch.allclose(outer, torch.outer(a, b))}')
print(f'  rank = {torch.linalg.matrix_rank(outer).item()}  (outer product is always rank-1)')

# r2: matrix-vector  Mx
mv = torch.einsum('ij,j->i', M, a)
print(f'Matrix-vector  shape {mv.shape}  match: {torch.allclose(mv, M @ a)}')

# r2: matrix-matrix  c_{ik} = Σ_j a_{ij} b_{jk}
mm = torch.einsum('ij,jk->ik', M, N)
print(f'Matrix-matrix  shape {mm.shape}  match: {torch.allclose(mm, M @ N)}')

# r2: Gram matrix  X'X  — element [j,k] = inner product of columns j and k
# Note: einsum 'ni,nj->ij' sums over observations n → gives X'X directly
gram_ein = torch.einsum('ni,nj->ij', M, M)   # M is (n=3, d=5) → 5×5 Gram
gram_ref = M.T @ M
print(f'Gram matrix    shape {gram_ein.shape}  match: {torch.allclose(gram_ein, gram_ref)}')

# Frobenius inner product  <M,P>_F = Σ_{ij} M_{ij} P_{ij}
frob = torch.einsum('ij,ij->', M, P)
print(f'Frobenius ip   {frob.item():.4f}  match: {torch.allclose(frob, (M*P).sum())}')

## Section 3 — Batch Operations

Adding a leading batch index `b` to any einsum string gives the batched version. This is how frameworks efficiently handle the mini-batch dimension without explicit Python loops.

In [ ]:
B, m, k, n = 8, 4, 6, 5
As = torch.randn(B, m, k)
Bs = torch.randn(B, k, n)
Sq = torch.randn(B, 4, 4)
X  = torch.randn(B, 3)
Y  = torch.randn(B, 3)

# batch matmul
bmm_ein = torch.einsum('bik,bkj->bij', As, Bs)
print(f'Batch matmul  shape {bmm_ein.shape}  match bmm: {torch.allclose(bmm_ein, torch.bmm(As, Bs))}')

# batch trace  — no single torch function; einsum is the idiomatic solution
btr_ein = torch.einsum('bii->b', Sq)
btr_ref = torch.stack([torch.trace(Sq[i]) for i in range(B)])
print(f'Batch trace   shape {btr_ein.shape}  match: {torch.allclose(btr_ein, btr_ref)}')

# batch outer product
bop_ein = torch.einsum('bi,bj->bij', X, Y)
bop_ref = torch.bmm(X.unsqueeze(2), Y.unsqueeze(1))
print(f'Batch outer   shape {bop_ein.shape}  match: {torch.allclose(bop_ein, bop_ref)}')

# batch diagonal extraction
bdiag_ein = torch.einsum('bii->bi', Sq)
print(f'Batch diag    shape {bdiag_ein.shape}  match: {torch.allclose(bdiag_ein, torch.diagonal(Sq,dim1=-2,dim2=-1))}')

## Section 4 — Quadratic Forms and Positive Definiteness &nbsp; `📖 r5`

From **Reading r5**: a symmetric matrix $A$ is **positive definite** iff $\mathbf{x}'A\mathbf{x} > 0$ for all nonzero $\mathbf{x}$. This quadratic form is a three-operand einsum contraction. The gradient of $f(\mathbf{x}) = \mathbf{x}'A\mathbf{x}$ is $(A+A')\mathbf{x}$, or $2A\mathbf{x}$ when $A$ is symmetric — you can verify this numerically with autograd.

In [ ]:
torch.manual_seed(0)
d = 5
G = torch.randn(d, d)
A_pd = G @ G.T + torch.eye(d)   # guaranteed PD: A = GG' + I
x = torch.randn(d)

# r5: quadratic form  x'Ax — three-operand contraction
qf_ein = torch.einsum('i,ij,j->', x, A_pd, x)
qf_ref = x @ A_pd @ x
print(f'Quadratic form  x\'Ax = {qf_ein.item():.4f}  match: {torch.allclose(qf_ein, qf_ref)}')
print(f'PD check (x\'Ax > 0 for random x): {qf_ein.item() > 0}')

# r5: verify PD via eigenvalues — all must be strictly positive
eigvals = torch.linalg.eigvalsh(A_pd)   # eigvalsh is for symmetric/Hermitian
print(f'Eigenvalues: {eigvals.round(decimals=3).tolist()}')
print(f'All positive (PD): {(eigvals > 0).all().item()}')

# r5: tr(A) = Σ λ_i  and  det(A) = Π λ_i
tr_ein = torch.einsum('ii->', A_pd)
print(f'\ntr(A) via einsum = {tr_ein.item():.4f},  Σ λ_i = {eigvals.sum().item():.4f}  match: {torch.allclose(tr_ein, eigvals.sum())}')
det_ref = torch.linalg.det(A_pd)
det_eig = eigvals.prod()
print(f'det(A) = {det_ref.item():.4f},  Π λ_i = {det_eig.item():.4f}  match: {torch.allclose(det_ref, det_eig, atol=1e-4)}')

## Section 5 — Spectral Decomposition &nbsp; `📖 r5`

From **Reading r5**, the **spectral theorem** says every symmetric matrix decomposes as:

$$A = Q\Lambda Q' = \sum_{k=1}^n \lambda_k \mathbf{q}_k \mathbf{q}_k'$$

The second form is the crucial insight: $A$ is a **sum of $n$ rank-1 matrices**, each being an outer product of an eigenvector with itself, scaled by the corresponding eigenvalue. This is exactly what einsum `'k,ik,jk->ij'` computes — and it makes explicit why you can truncate to the top $r$ eigenvalues to get the best rank-$r$ approximation.

In [ ]:
torch.manual_seed(1)
n = 6
G2 = torch.randn(n, n)
A_sym = G2 @ G2.T   # symmetric PSD matrix

# r5: eigendecomposition — eigvalsh gives real eigenvalues for symmetric matrices
lam, Q = torch.linalg.eigh(A_sym)   # lam sorted ascending, Q orthonormal columns
print(f'Eigenvalues (ascending): {lam.round(decimals=2).tolist()}')
print(f'Q orthonormal: QQ\' ≈ I?  max err = {(Q @ Q.T - torch.eye(n)).abs().max().item():.2e}')

# r5: reconstruct A = Q Λ Q'  via einsum
# String 'k,ik,jk->ij': sum over eigenvectors k of  λ_k * q_{ik} * q_{jk}
# This is exactly  Σ_k λ_k q_k q_k'  — the rank-1 outer-product expansion
A_recon = torch.einsum('k,ik,jk->ij', lam, Q, Q)
print(f'\nSpectral reconstruction  A = Σ λ_k q_k q_k\'')
print(f'  Max reconstruction error: {(A_recon - A_sym).abs().max().item():.2e}')
print(f'  Matches A: {torch.allclose(A_recon, A_sym, atol=1e-5)}')

# Equivalent to Q @ diag(lam) @ Q.T — show they match
A_matmul = Q @ torch.diag(lam) @ Q.T
print(f'  Same as Q diag(λ) Q\': {torch.allclose(A_recon, A_matmul, atol=1e-5)}')

# r5: examine individual rank-1 terms  λ_k * q_k q_k'
print(f'\nRank-1 components (sorted by eigenvalue magnitude):')
idx = torch.argsort(lam, descending=True)
for i in range(3):
    k = idx[i]
    term = lam[k] * torch.outer(Q[:, k], Q[:, k])  # λ_k q_k q_k'
    term_ein = torch.einsum('i,j->', Q[:, k], Q[:, k]) * lam[k]  # scalar check
    frob = torch.linalg.norm(term, 'fro').item()
    print(f'  k={i}: λ={lam[k].item():.3f}  ||λ_k q_k q_k\'||_F = {frob:.3f}  rank={torch.linalg.matrix_rank(term).item()}')

# r5: partial reconstruction with top-r terms
for r in [1, 2, 4, n]:
    top_idx = idx[:r]
    A_r = torch.einsum('k,ik,jk->ij', lam[top_idx], Q[:, top_idx], Q[:, top_idx])
    err = torch.linalg.norm(A_sym - A_r, 'fro').item()
    var_explained = lam[top_idx].sum().item() / lam.sum().item()
    print(f'  r={r}: Frobenius error={err:.3f},  variance explained={var_explained:.1%}')

## Section 6 — SVD and Truncated Low-Rank Approximation &nbsp; `📖 r5`

From **Reading r5**, the SVD generalises the eigendecomposition to *rectangular* matrices:

$$A = U\Sigma V', \qquad A_r = \sum_{i=1}^r \sigma_i \mathbf{u}_i \mathbf{v}_i'$$

The **Eckart–Young theorem** says $A_r$ is the closest rank-$r$ matrix to $A$ in the Frobenius norm, and the approximation error equals $\sqrt{\sum_{i>r} \sigma_i^2}$. The einsum string `'r,mr,nr->mn'` computes this sum of outer products directly.

In [ ]:
torch.manual_seed(2)
m_svd, n_svd = 10, 8
A_rect = torch.randn(m_svd, n_svd)

# r5: SVD  A = U Σ V'
U, S, Vh = torch.linalg.svd(A_rect, full_matrices=False)  # U:(m,k) S:(k,) Vh:(k,n)
print(f'A shape: {A_rect.shape}')
print(f'U: {U.shape}  S: {S.shape}  Vh: {Vh.shape}')
print(f'Singular values: {S.round(decimals=2).tolist()}')

# r5: full reconstruction  A = Σ_i σ_i u_i v_i'  via einsum
# String 'r,mr,rn->mn': for each r, outer product u_r v_r' scaled by σ_r, then sum
A_full = torch.einsum('r,mr,rn->mn', S, U, Vh)
print(f'\nFull SVD reconstruction  A = Σ σ_i u_i v_i\'')
print(f'  Max error: {(A_full - A_rect).abs().max().item():.2e}')
print(f'  Match: {torch.allclose(A_full, A_rect, atol=1e-5)}')

# r5: Eckart-Young truncated approximation at rank r
print(f'\nEckart-Young rank-r approximation (Frobenius error = √Σ_{{i>r}} σ_i²):')
print(f'{"r":>4} {"||A-A_r||_F":>14} {"predicted":>14} {"σ₁/σᵣ":>10} {"% energy":>10}')
total_energy = (S ** 2).sum().item()
for r in [1, 2, 3, 4, 6, 8]:
    A_r  = torch.einsum('r,mr,rn->mn', S[:r], U[:, :r], Vh[:r, :])
    err_actual    = torch.linalg.norm(A_rect - A_r, 'fro').item()
    err_predicted = ((S[r:] ** 2).sum() ** 0.5).item() if r < len(S) else 0.0
    energy_kept   = (S[:r] ** 2).sum().item() / total_energy
    ratio = (S[0] / S[r-1]).item() if r > 0 else 1.0
    print(f'{r:>4} {err_actual:>14.4f} {err_predicted:>14.4f} {ratio:>10.2f} {energy_kept:>9.1%}')

print('\n(Actual error = predicted error confirms the Eckart-Young theorem.)')

# r5: σ_i = sqrt(eigenvalue of A'A)
ATA_eigvals = torch.linalg.eigvalsh(A_rect.T @ A_rect)
sigma_from_eig = ATA_eigvals.clamp(min=0).sqrt().flip(0)  # sort descending
k = min(m_svd, n_svd)
print(f'\nσ_i vs √(eigenvalues of A\'A)  match: {torch.allclose(S, sigma_from_eig[:k], atol=1e-4)}')

## Section 7 — PCA via Covariance Eigendecomposition &nbsp; `📖 r5`

From **Reading r5**, the full PCA pipeline is:
1. Form the sample covariance $S = \frac{1}{n-1}X'X$ (with $X$ mean-centred) — this is an einsum
2. Eigendecompose $S = Q\Lambda Q'$ — eigenvectors are the principal components
3. Project $Z = XQ_r$ — reduce to $r$ dimensions — this is another einsum
4. Explained variance ratio: $\lambda_k / \sum_j \lambda_j$

Each 3DGS Gaussian's scale parameters $(s_x, s_y, s_z)$ are the square roots of the eigenvalues of its covariance — they are the lengths of the three principal semi-axes of the ellipsoid.

In [ ]:
torch.manual_seed(3)

# Synthetic data: 200 samples, 8 features, with known low-dimensional structure
n_obs, n_feat = 200, 8
true_rank = 3
Z_true = torch.randn(n_obs, true_rank)                     # 3 latent dimensions
W      = torch.randn(true_rank, n_feat)                    # mixing matrix
X_raw  = Z_true @ W + 0.2 * torch.randn(n_obs, n_feat)    # observed features

# Step 1: mean-centre
X = X_raw - X_raw.mean(dim=0)

# Step 2: sample covariance  S = (1/(n-1)) X'X  via einsum
# 'ni,nj->ij' sums over observations n, giving the (n_feat × n_feat) covariance
S_ein = torch.einsum('ni,nj->ij', X, X) / (n_obs - 1)
S_ref = X.T @ X / (n_obs - 1)
print(f'Covariance S  shape={S_ein.shape}  match: {torch.allclose(S_ein, S_ref)}')
print(f'S symmetric:  {torch.allclose(S_ein, S_ein.T)}')

# Step 3: eigendecompose S = Q Λ Q'  (eigvalsh for symmetric, returns ascending order)
lam_pca, Q_pca = torch.linalg.eigh(S_ein)

# Sort descending (largest variance first, as in standard PCA convention)
idx  = torch.argsort(lam_pca, descending=True)
lam_pca = lam_pca[idx]
Q_pca   = Q_pca[:, idx]   # columns are principal components

print(f'\nEigenvalues (variance per PC): {lam_pca.round(decimals=3).tolist()}')

# Explained variance ratio
total_var = lam_pca.sum()
explained = lam_pca / total_var
print(f'Explained variance ratio:       {[f"{v:.3f}" for v in explained.tolist()]}')

# Step 4: project onto top-r PCs  Z = X Q_r  via einsum 'nd,dr->nr'
r_pca = 3
Q_r = Q_pca[:, :r_pca]   # top-r eigenvectors  (n_feat × r)
Z_pca = torch.einsum('nd,dr->nr', X, Q_r)
Z_ref = X @ Q_r
print(f'\nPCA projection  Z = X Q_r  shape={Z_pca.shape}  match: {torch.allclose(Z_pca, Z_ref)}')
print(f'Cumulative variance (top 3 PCs): {explained[:r_pca].sum().item():.1%}')

# Verify: covariance of Z is diagonal (PCs are uncorrelated)
S_Z = torch.einsum('ni,nj->ij', Z_pca, Z_pca) / (n_obs - 1)
off_diag_max = S_Z.fill_diagonal_(0).abs().max().item()
print(f'\nCovariance of projected Z — max off-diagonal: {off_diag_max:.2e}  (should be ≈ 0, PCs are uncorrelated)')

# r5: 3DGS link — eigenvalues of Gaussian covariance = squared scales
# If the covariance of one Gaussian is Sigma, its scale parameters are sqrt(eigenvalues)
Sigma_single = torch.diag(torch.tensor([4.0, 1.0, 0.25]))  # diagonal example
eigs_sigma = torch.linalg.eigvalsh(Sigma_single)
scales = eigs_sigma.clamp(min=0).sqrt()
print(f'\n3DGS link: Σ = diag(4, 1, 0.25) → scales (s_x, s_y, s_z) = {scales.tolist()}')
print('  These are the lengths of the Gaussian ellipsoid semi-axes.')

## Section 8 — Batch 3DGS Covariance $\Sigma = R\,\text{diag}(s^2)\,R'$ &nbsp; `📖 r5 + 3DGS`

From **Reading r5**: the 3DGS covariance $\Sigma = RSS'R'$ is PSD by construction. Since $S$ is diagonal, $SS' = \text{diag}(s_1^2, s_2^2, s_3^2)$, so element-wise:

$$\Sigma_{ij} = \sum_k R_{ik}\, s_k^2\, R_{jk}$$

This is a batched three-operand einsum — the same structure as the spectral decomposition $A = \sum_k \lambda_k q_{ik} q_{jk}$, except here the "eigenvectors" are the columns of $R$ (rotation axes) and the "eigenvalues" are $s_k^2$.

In [ ]:
def random_rotation_batch(N):
    """Random orthogonal matrices via QR (det=+1)."""
    Q, R = torch.linalg.qr(torch.randn(N, 3, 3))
    Q = Q * torch.diagonal(R, dim1=-2, dim2=-1).sign().unsqueeze(-2)
    return Q

N_gauss = 5000
R_batch = random_rotation_batch(N_gauss)           # (N, 3, 3) — each column is a principal axis
s_batch = torch.exp(torch.randn(N_gauss, 3) - 1)  # positive scales, log-normal
s2      = s_batch ** 2                             # (N, 3) — squared scales = eigenvalues of Sigma

# r5 connection: Σ_n = Σ_k s²_{n,k} * r_{n,·,k} r_{n,·,k}'  (sum of 3 rank-1 terms)
# Same form as spectral decomp: A = Σ_k λ_k q_k q_k'
# Einsum 'nik,nk,njk->nij': for each Gaussian n and axis k, outer product of column k of R
Sigma = torch.einsum('nik,nk,njk->nij', R_batch, s2, R_batch)

# Reference (explicit loop, obviously correct)
Sigma_ref = torch.stack([
    R_batch[n] @ torch.diag(s2[n]) @ R_batch[n].T for n in range(N_gauss)
])

print(f'Covariance shape : {Sigma.shape}')
print(f'Match reference  : {torch.allclose(Sigma, Sigma_ref, atol=1e-5)}')

# r5: verify PSD — all eigenvalues of each Σ_n must be ≥ 0
eigvals_all = torch.linalg.eigvalsh(Sigma)   # (N, 3), ascending
print(f'Min eigenvalue   : {eigvals_all.min().item():.2e}  (≥ 0 → PSD)')
print(f'All PSD          : {(eigvals_all >= -1e-6).all().item()}')

# r5: the recovered eigenvalues should equal s² (since R is orthogonal)
# eigvalsh returns ascending; s2 is unsorted, so compare sorted values
eigvals_sorted   = eigvals_all.sort(dim=-1).values
s2_sorted        = s2.sort(dim=-1).values
print(f'Eigenvalues ≈ s² : {torch.allclose(eigvals_sorted, s2_sorted, atol=1e-4)}')
print('  (confirmed: the stored scale params are exactly the Gaussian semi-axis lengths)')

## Section 9 — ML Pattern: Scaled Dot-Product Attention

The two core einsum calls in the transformer attention mechanism. The QK score is a batched inner product across the head dimension; the output is a batched outer product accumulation.

In [ ]:
Bsz, H, S_seq, Dh = 2, 4, 6, 16
Q_att = torch.randn(Bsz, H, S_seq, Dh)
K_att = torch.randn(Bsz, H, S_seq, Dh)
V_att = torch.randn(Bsz, H, S_seq, Dh)

# Step 1: QK scores — inner product over d_head for each (batch, head, query, key) pair
scores = torch.einsum('bhid,bhjd->bhij', Q_att, K_att) * (Dh ** -0.5)
scores_ref = (Q_att @ K_att.transpose(-2, -1)) * (Dh ** -0.5)
print(f'Scores  shape={scores.shape}  match: {torch.allclose(scores, scores_ref)}')

# Step 2: softmax weights
w = torch.softmax(scores, dim=-1)

# Step 3: weighted sum of values — outer product accumulation over sequence key dim
out = torch.einsum('bhij,bhjd->bhid', w, V_att)
out_ref = w @ V_att
print(f'Output  shape={out.shape}  match: {torch.allclose(out, out_ref)}')

# These two einsum strings appear verbatim in PyTorch's F.scaled_dot_product_attention
# and in every transformer implementation.

## Section 10 — Performance Benchmark

On large tensors, `torch.einsum` dispatches to the same cuBLAS / oneDNN kernels as `torch.bmm` and `torch.matmul`. The overhead for parsing the equation string is negligible.

In [ ]:
def bench(fn, *args, warmup=10, iters=100):
    for _ in range(warmup): fn(*args)
    if device == 'cuda': torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters): fn(*args)
    if device == 'cuda': torch.cuda.synchronize()
    return (time.perf_counter() - t0) / iters * 1e3

# Batch matmul
B2, M2, K2, N2 = 64, 128, 128, 128
As2 = torch.randn(B2, M2, K2, device=device)
Bs2 = torch.randn(B2, K2, N2, device=device)
t_ein = bench(lambda: torch.einsum('bik,bkj->bij', As2, Bs2))
t_bmm = bench(lambda: torch.bmm(As2, Bs2))
print(f'Batch matmul ({B2}×{M2}×{K2}) × ({B2}×{K2}×{N2})')
print(f'  einsum : {t_ein:.3f} ms  |  bmm : {t_bmm:.3f} ms  |  ratio : {t_ein/t_bmm:.2f}x')

# Attention QK scores
B3, H3, S3, Dh3 = 8, 8, 128, 64
Q3 = torch.randn(B3, H3, S3, Dh3, device=device)
K3 = torch.randn(B3, H3, S3, Dh3, device=device)
t_ein_a = bench(lambda: torch.einsum('bhid,bhjd->bhij', Q3, K3))
t_mat_a = bench(lambda: (Q3 @ K3.transpose(-2,-1)))
print(f'\nAttention QK ({B3}×{H3}×{S3}×{Dh3})')
print(f'  einsum : {t_ein_a:.3f} ms  |  matmul : {t_mat_a:.3f} ms  |  ratio : {t_ein_a/t_mat_a:.2f}x')
print('\nRatios near 1.0 = einsum has negligible overhead over explicit ops.')

## Key Takeaways

**From the readings, now in code:**

| Reading formula | Einsum string | Section |
|----------------|--------------|--------|
| $b_{ij} = a_{ji}$ (r2 transpose) | `'ij->ji'` | 1 |
| $\mathbf{a}'\mathbf{b} = \sum_i a_i b_i$ (r2 inner product) | `'i,i->'` | 2 |
| $c_{ik} = \sum_j a_{ij} b_{jk}$ (r2 matrix multiply) | `'ij,jk->ik'` | 2 |
| $[X'X]_{jk}$ (r2 Gram / covariance) | `'ni,nj->ij'` | 2, 7 |
| $\mathbf{x}'A\mathbf{x}$ (r5 quadratic form, PD) | `'i,ij,j->'` | 4 |
| $A = \sum_k \lambda_k \mathbf{q}_k\mathbf{q}_k'$ (r5 spectral decomp) | `'k,ik,jk->ij'` | 5 |
| $A_r = \sum_{i=1}^r \sigma_i \mathbf{u}_i\mathbf{v}_i'$ (r5 truncated SVD) | `'r,mr,rn->mn'` | 6 |
| $Z = XQ_r$ (r5 PCA projection) | `'nd,dr->nr'` | 7 |
| $\Sigma = R\,\text{diag}(s^2)\,R'$ (r5 + 3DGS) | `'nik,nk,njk->nij'` | 8 |

- The spectral and 3DGS covariance einsum strings are **identical in structure** — both are $\sum_k \text{eigenvalue}_k \cdot \text{outer}(\text{eigenvec}_k, \text{eigenvec}_k)$.
- **Framework-portable**: these exact strings work in `np.einsum`, `tf.einsum`, and `jnp.einsum`.